# Elastoplasticity Informed Kolmogorov-Arnold Networks Using Chebyshev Polynomials (EPi-cKAN)

**Paper:** Mostajeran, F., Faroughi, S.A. (2026). *Elastoplasticity Informed Kolmogorov-Arnold Networks Using Chebyshev Polynomials.* International Journal for Numerical and Analytical Methods in Geomechanics, 2026;0:1-24. https://doi.org/10.1002/nag.70283 (preprint: arXiv:2410.10897, "EPi-cKANs: Elasto-Plasticity Informed Kolmogorov-Arnold Networks Using Chebyshev Polynomials").

**Carpeta origen:** `Ciencia, energía nuclear y química/Elastoplasticity_Informed_Kolmogorov-Arnold_Networ.pdf`

## Como se usan las KAN en este paper

Este paper presenta EPi-cKAN, una red Kolmogorov-Arnold basada en polinomios de Chebyshev (cKAN) e informada por la fisica de la elastoplasticidad, pensada para sustituir al modelo constitutivo clasico de un material granular (arena) bajo carga triaxial. El objetivo es predecir el incremento del indice de vacios Δe, el incremento de deformacion plastica Δε^p_ij y el incremento de tension Δσ_ij a partir del estado actual del material (indice de vacios e, tension σ_ij, deformacion total ε_ij, deformacion plastica ε^p_ij) y de un incremento de deformacion impuesto Δε_ij.

### Capa KAN con polinomios de Chebyshev (cKAN)

En vez de las B-splines de la formulacion original de KAN (Liu et al. 2024, ya usada en `pykan`), este paper reemplaza cada funcion univariable de arista por una expansion en polinomios de Chebyshev de primera especie T_n(x), definidos por la recurrencia

$$T_0(x)=1,\quad T_1(x)=x,\quad T_n(x)=2xT_{n-1}(x)-T_{n-2}(x),\ n\ge2$$

$$\phi(x)=\sum_i c_i T_i(x)\qquad\text{(Eq. 25)}$$

con $c_i$ coeficientes entrenables. Esto sustituye la funcion $\phi(x)=w_b b(x)+w_s\,\text{spline}(x)$ de la KAN original (Eq. 22-23) y reduce el numero de parametros de $O(n_lH^2(k+g))$ (con $g$ puntos de grid de la spline) a $O(n_lH^2k)$ (sin grid), con $k$ el orden del polinomio. Al no depender de un grid, `pykan` (basada en B-splines) no reproduce exactamente esta arquitectura, asi que este cuaderno implementa una capa `ChebyKANLayer` propia en PyTorch.

### Arquitectura EPi-cKAN: fisica integrada en la red y en la perdida

La arquitectura (Fig. 3b del paper) consta de tres sub-redes cKAN interconectadas:

- **cKAN$^{\varepsilon^p}$**: predice el incremento de deformacion plastica $\Delta\varepsilon^p$ a partir de $(e,\sigma_{ij},\varepsilon_{ij},\varepsilon^p_{ij},\Delta\varepsilon_{ij})$.
- **cKAN$^{e}$**: predice el incremento del indice de vacios $\Delta e$, con las mismas entradas.
- **cKAN$^{\sigma}$**: en vez de predecir $\Delta\sigma_{ij}$ directamente, predice una estimacion $\tilde G$ del modulo de corte secante. Junto con un escalar $\tilde R$ estimado de forma independiente, esto construye el tensor de elasticidad secante $\tilde C_{ijkl}$ (misma forma que la Eq. 8), y el incremento de tension se obtiene mediante una capa fisica no entrenable (Eq. 27):

$$\Delta\sigma_{ij}=\tilde C_{ijkl}:(\Delta\varepsilon_{kl}-\Delta\varepsilon^{p}_{kl})$$

donde $\Delta\varepsilon^p_{kl}$ es la salida **predicha** por cKAN$^{\varepsilon^p}$ en el mismo forward pass. Esta es la pieza central "informada por la fisica" del modelo: en vez de dejar que la red aprenda $\Delta\sigma$ como funcion arbitraria de las entradas, se le impone la forma funcional exacta de la ley elastica hipoelastica del modelo WG, y solo el modulo $\tilde G$ (y el escalar $\tilde R$) quedan libres para el aprendizaje. Ademas, cKAN$^\sigma$ usa una base "aumentada" (Eq. 29), $\phi(x)=\Gamma(x)+\text{Chebyshev}(x)$ con $\Gamma(x)=\omega\cdot\text{ReLU}(x)$, que combina una funcion base tipo ReLU ponderada (pesos $\omega$ entrenables) con la expansion de Chebyshev.

La funcion de costo total (Eq. 26 y 28) es

$$CF=CF^e+CF^{\varepsilon^p}+CF^\sigma,\quad CF^e=\text{MSE}(\Delta e,\Delta e^*),\ \ CF^{\varepsilon^p}=\text{MSE}(\Delta\varepsilon^p,\Delta\varepsilon^{p*}),\ \ CF^\sigma=\text{MSE}\big(\tilde C:(\Delta\varepsilon-\Delta\varepsilon^p),\Delta\sigma^*\big)$$

Al entrenar las tres sub-redes de forma simultanea con esta perdida conjunta, los errores en la prediccion de la deformacion plastica se propagan -- via la capa fisica de la Eq. 27 -- al error de tension, acoplando fisicamente las tres salidas. Esto distingue a EPi-cKAN de un cKAN puramente "paralelo" (Fig. 1b del paper), que entrena tres sub-redes independientes sin esta relacion fisica explicita, y del EPNN (version MLP del mismo principio, Eghbalian et al. 2023) contra el que tambien se compara en el paper.

### Fisica de la elastoplasticidad: modelo Wan-Guo (WG) para arena

El modelo WG (Wan y Guo), formulado en el marco de Mohr-Coulomb con teoria del estado critico, describe el comportamiento de arenas bajo pequenas deformaciones y es el que se usa para generar los datos (sinteticos, con control total de las variables internas):

- Descomposicion de la deformacion: $\varepsilon_{ij}=\varepsilon^e_{ij}+\varepsilon^p_{ij}$ (Eq. 1)
- Elasticidad hipoelastica con modulo de corte dependiente del estado: $G=G^0\frac{(2.17-e)^2}{1+e}\sqrt{p^0p}$, $R=\frac{2(1+\nu)}{3(1-2\nu)}$ (Eq. 9), tensor $C_{ijkl}=G\big[(R-\tfrac23)\delta_{ij}\delta_{kl}+\delta_{ik}\delta_{jl}+\delta_{il}\delta_{jk}\big]$ (Eq. 8)
- Superficie de fluencia $F=q-Mp\le0$ y potencial plastico no asociado $P=q-Np$ (Eq. 10), con $p=\text{tr}(\sigma)/3$ y $q=\sqrt{\tfrac32 s_{ij}s_{ij}}$ los invariantes de tension media y desviadora (Eq. 11)
- Regla de flujo plastico $\dot\varepsilon^p_{ij}=\dot\lambda\,\partial P/\partial\sigma_{ij}$ (Eq. 5) sujeta a las condiciones de Kuhn-Tucker $\dot\lambda\ge0,\ F\le0,\ \dot\lambda F=0$ (Eq. 6)
- Endurecimiento del angulo de friccion movilizado y evolucion del indice de vacios critico con la deformacion plastica de corte $\gamma^p=\sqrt{\tfrac23\eta^p_{ij}\eta^p_{ij}}$ y la razon de vacios (Eq. 14-15), y evolucion cinematica del indice de vacios $\dot e=-(1+e)\dot\varepsilon^v$ (Eq. 16)

Como el dataset original (generado por integracion numerica implicita -- Euler regresivo con return mapping -- del modelo WG para arena de Ottawa, Tabla 1 del paper) no es publico, este cuaderno **genera sus propios datos sinteticos**, integrando explicitamente (con sub-pasos finos) estas mismas ecuaciones fisicas con los parametros de arena de Ottawa de la Tabla 1(a) del paper. Las simplificaciones respecto al articulo original -- necesarias para que el modelo fisico de referencia quepa en un cuaderno razonable -- se documentan en la celda "Nota honesta sobre los resultados".

## Repositorio publico

El *Data Availability Statement* del paper dice unicamente: "The data that support the findings of this study are available from the corresponding author upon reasonable request" -- no se referencia ningun repositorio publico de codigo ni de datos en el texto, ni en Acknowledgments. Una busqueda adicional en GitHub (autores, Faroughi Lab, Universidad de Utah) tampoco encontro un repositorio publico asociado a EPi-cKAN.

- **No se encontro repositorio publico.** La implementacion de este cuaderno (capa `ChebyKANLayer`, arquitectura EPi-cKAN, generador de datos WG y bucle de entrenamiento) se hizo **desde cero**, siguiendo las ecuaciones del paper (Eqs. 1-29) y usando `pykan` (ya clonado en `Kolmogorov-Arnold Networks/codigo/pykan`, version 0.2.8) solo como referencia conceptual de la libreria KAN de base (B-splines), no como codigo fuente directo, ya que el cKAN de este paper usa una base distinta (Chebyshev, Eq. 25).

In [ ]:
%pip install -q torch numpy matplotlib

## 1. Fisica del modelo Wan-Guo (WG) y generador de datos sinteticos

Como el dataset del paper no es publico, generamos aqui nuestros propios datos integrando el modelo WG con los parametros de arena de Ottawa (Tabla 1a). Para mantener el cuaderno manejable, se trabaja en el subespacio **axisimetrico triaxial** (deformaciones principales $\varepsilon_{11}=\varepsilon_{22}$, $\varepsilon_{33}$), representando el estado del material con los invariantes $(e,p,q,\varepsilon_v,\varepsilon_q,\varepsilon^{v,p},\gamma^p)$ en vez del tensor completo de 6 componentes -- exactamente el mismo subespacio en el que el propio paper reporta todos sus resultados (Figs. 9-14).

El integrador (`wg_substep`) avanza el estado con **sub-pasos explicitos y multiplicador plastico consistente**: en cada sub-paso resolvemos $\dot\lambda$ imponiendo la condicion de consistencia $\dot F=0$ sobre $F=q-Mp$ de forma cerrada,

$$\dot\lambda=\frac{3G\,\dot\varepsilon_q-MK\,\dot\varepsilon_v}{3G+p\,\partial M/\partial\gamma^p+MKN}\ \ge 0\quad\text{(Kuhn-Tucker, Eq. 6)}$$

con $K=R\,G$ (Eq. 9), $M$ el coeficiente de friccion movilizado (Eq. 14b, exacto del paper) y $N$ la dilatancia. Esto reemplaza el Euler regresivo con *return mapping* del paper (Eq. no numerada en Sec. 2.3) por un esquema explicito equivalente para trayectorias suaves, mucho mas simple de implementar de forma robusta en un cuaderno.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
import matplotlib.pyplot as plt
import time

t_start = time.time()
torch.manual_seed(0)
np.random.seed(0)
device = torch.device('cpu')
print('Device:', device)

# --- Parametros de arena de Ottawa (Tabla 1a del paper) ---
G0, nu = 900.0, 0.30          # kPa, Poisson
sin_psi_cs, n_m, alpha0 = 0.53, 1.3, 0.008
e_cs0, d_cs, n_cs = 0.74, 0.005, 0.40
p0 = 1.0                       # kPa, tension media de referencia
n_d = 0.2                      # factor de amortiguamiento de la dilatancia (ver Nota honesta)

R_const = 2 * (1 + nu) / (3 * (1 - 2 * nu))          # Eq. 9
M_cs = 6 * sin_psi_cs / (3 - sin_psi_cs)             # razon de tension critica (triax. compresion)
print(f'R = {R_const:.4f}   M_cs = {M_cs:.4f}')


def shear_modulus(e, p):
    """G(e,p), Eq. 9."""
    return G0 * (2.17 - e) ** 2 / (1 + e) * np.sqrt(np.maximum(p0 * p, 1e-8))


def e_critical(p):
    """Indice de vacios critico e_cs(p), Eq. 14a."""
    return e_cs0 * np.exp(-d_cs * (np.maximum(p, 1e-6) / p0) ** n_cs)


def mobilized_M(e, p, gammap):
    """Angulo de friccion movilizado -> M=6 sin(psi)/(3-sin(psi)), Eq. 14b + 13 (M^tc)."""
    ecs = e_critical(p)
    frac = gammap / (alpha0 + gammap) if gammap > 0 else 0.0
    sin_psi = (e / ecs) ** (-n_m) * frac * sin_psi_cs
    sin_psi = float(np.clip(sin_psi, -0.99, 0.99))
    M = 6 * sin_psi / (3 - sin_psi)
    return M, sin_psi


def wg_substep(state, dev, deq, n_sub=5):
    """Avanza el estado un incremento de deformacion (dev, deq) del modelo WG,
    integrando de forma explicita en n_sub sub-pasos con el multiplicador
    plastico consistente (tangente), ver celda anterior."""
    dev_s, deq_s = dev / n_sub, deq / n_sub
    for _ in range(n_sub):
        e, p, q, gammap = state['e'], state['p'], state['q'], state['gammap']
        G = shear_modulus(e, p)
        K = R_const * G
        M, sin_psi = mobilized_M(e, p, gammap)
        eta = q / p if p > 1e-6 else 0.0
        N = n_d * (eta - M_cs)                        # dilatancia (Rowe/estado critico simplificada)
        h = 1e-6
        M_plus, _ = mobilized_M(e, p, gammap + h)
        dM_dgamma = (M_plus - M) / h
        denom = 3 * G + dM_dgamma * p + M * K * N
        dlambda = 0.0 if abs(denom) < 1e-8 else (3 * G * deq_s - M * K * dev_s) / denom
        dlambda = max(dlambda, 0.0)                    # Kuhn-Tucker: multiplicador >= 0
        devp, deqp = -N * dlambda, dlambda
        deve, deqe = dev_s - devp, deq_s - deqp
        dp, dq = K * deve, 3 * G * deqe
        de = -(1 + e) * dev_s                          # Eq. 16
        state['e'] = e + de
        state['p'] = max(p + dp, 1.0)
        state['q'] = max(q + dq, 0.0)
        state['ev'] += dev_s
        state['eq'] += deq_s
        state['evp'] += devp
        state['gammap'] += deqp
    return state


def simulate_path(xi, p_in, e_in, n_steps=70, rng=None, random_incr=True, eps33_final=0.07):
    """Simula una trayectoria proporcional axisimetrica completa: eps11=eps22=eps33/xi.
    Con random_incr=True, el incremento axial por paso es aleatorio en [0,0.0016]
    (igual que en la Seccion 2.3.1 del paper); con False, es constante = eps33_final/n_steps
    (igual que en la Seccion 4.2, trayectoria ciega)."""
    if rng is None:
        rng = np.random
    state = dict(e=e_in, p=p_in, q=0.0, ev=0.0, eq=0.0, evp=0.0, gammap=0.0)
    records = [state.copy()]
    for _ in range(n_steps):
        d33 = rng.uniform(0.0, 0.0016) if random_incr else eps33_final / n_steps
        d11 = d33 / xi
        dev = 2 * d11 + d33
        deq = (2.0 / 3.0) * (d33 - d11)
        state = wg_substep(state, dev, deq, n_sub=5)
        records.append(state.copy())
    return records


# comprobacion rapida: una trayectoria de referencia (xi=-2.5, la usada en la Sec. 4.2 del paper)
recs_check = simulate_path(-2.5, 375.0, 0.64, n_steps=100, random_incr=False)
print('e final:', recs_check[-1]['e'], ' p final (kPa):', recs_check[-1]['p'],
      ' q final (kPa):', recs_check[-1]['q'])

## 2. Generacion del dataset de entrenamiento

Siguiendo la Seccion 2.3.1 del paper: se generan multiples trayectorias proporcionales aleatorias, cada una con una relacion $\xi=\varepsilon_{33}/\varepsilon_{11}$ (analoga a la SPD -- *strain path direction* -- del paper, restringida aqui al subespacio axisimetrico), una presion de confinamiento inicial $p^{in}$ y un indice de vacios inicial $e^{in}$ muestreados dentro de los rangos de la Tabla 1(b) del paper ($p^{in}\in[50,500]$ kPa, $e^{in}\in[0.5,0.74]$), y con incrementos de deformacion axial aleatorios entre 0.0 y 0.0016 en cada paso. Cada par de estados consecutivos $(s_n,s_{n+1})$ de cada trayectoria produce una muestra de entrenamiento:

- **Entrada** ($x$, 9 componentes): $(e,p,q,\varepsilon_v,\varepsilon_q,\varepsilon^{v,p},\gamma^p,\Delta\varepsilon_v,\Delta\varepsilon_q)$ en el paso $n$ -- version reducida del $x=\{e,\sigma_{ij},\varepsilon_{ij},\varepsilon^p_{ij},\Delta\varepsilon_{ij}\}$ del paper.
- **Salida** ($y$, 5 componentes): $(\Delta e,\Delta\varepsilon^{v,p},\Delta\gamma^p,\Delta p,\Delta q)$ entre los pasos $n$ y $n+1$.

Se divide el dataset en 70% entrenamiento / 15% validacion / 15% evaluacion, y se normalizan las 9 entradas a $[-1,1]$ por min-max (igual que en la Seccion 2.3 del paper).

In [ ]:
def build_dataset(n_paths=180, n_steps=70, seed=0):
    rng = np.random.RandomState(seed)
    X, Y = [], []
    for _ in range(n_paths):
        xi = -rng.uniform(1.2, 30.0)          # familia de trayectorias de compresion triaxial
        p_in = rng.uniform(50.0, 500.0)       # Tabla 1(b)
        e_in = rng.uniform(0.50, 0.74)        # Tabla 1(b)
        recs = simulate_path(xi, p_in, e_in, n_steps=n_steps, rng=rng, random_incr=True)
        for i in range(len(recs) - 1):
            s0, s1 = recs[i], recs[i + 1]
            dev, deq = s1['ev'] - s0['ev'], s1['eq'] - s0['eq']
            x = [s0['e'], s0['p'], s0['q'], s0['ev'], s0['eq'], s0['evp'], s0['gammap'], dev, deq]
            y = [s1['e'] - s0['e'], s1['evp'] - s0['evp'], s1['gammap'] - s0['gammap'],
                 s1['p'] - s0['p'], s1['q'] - s0['q']]
            X.append(x); Y.append(y)
    return np.array(X, dtype=np.float32), np.array(Y, dtype=np.float32)


X, Y = build_dataset(n_paths=180, n_steps=70, seed=0)
print('Dataset:', X.shape, Y.shape)

n = X.shape[0]
idx = np.random.RandomState(1).permutation(n)
n_train, n_val = int(0.7 * n), int(0.15 * n)
tr, va, te = idx[:n_train], idx[n_train:n_train + n_val], idx[n_train + n_val:]
X_train, Y_train = X[tr], Y[tr]
X_val, Y_val = X[va], Y[va]
X_test, Y_test = X[te], Y[te]

x_min, x_max = X_train.min(axis=0), X_train.max(axis=0)


def normalize_x(Xraw):
    return 2 * (Xraw - x_min) / (x_max - x_min + 1e-12) - 1


# desviaciones tipicas de cada salida, usadas para balancear la perdida CF = CF^e+CF^eps^p+CF^sigma
# (necesario porque Delta e, Delta eps^v,p ~ 1e-4 y Delta p, Delta q ~ decenas/cientos de kPa)
target_std = Y_train.std(axis=0) + 1e-8
target_std_t = torch.tensor(target_std, dtype=torch.float32)

Xt_train, Xnt_train, Yt_train = torch.tensor(X_train), torch.tensor(normalize_x(X_train)), torch.tensor(Y_train)
Xt_val, Xnt_val, Yt_val = torch.tensor(X_val), torch.tensor(normalize_x(X_val)), torch.tensor(Y_val)
Xt_test, Xnt_test, Yt_test = torch.tensor(X_test), torch.tensor(normalize_x(X_test)), torch.tensor(Y_test)

## 3. Capa cKAN (Chebyshev) y arquitectura EPi-cKAN

`ChebyKANLayer` implementa $\phi(x)=\sum_i c_iT_i(x)$ (Eq. 25) con la recurrencia de Chebyshev, normalizando la entrada a $[-1,1]$ con `tanh` antes de evaluar los polinomios (dominio natural de $T_i$). `AugmentedChebyKANLayer` anade el termino $\Gamma(x)=\omega\cdot\text{ReLU}(x)$ de la Eq. 29, usado solo en la ultima capa de la sub-red $\text{cKAN}^\sigma$.

`EPiCKAN` ensambla las tres sub-redes con las mismas dimensiones $(L,N,k)$ de la Tabla 2 del paper para la version EPi-cKAN -- $(2,20,3)$ para $e$, $(3,20,4)$ para $\varepsilon^p$ y $(3,20,4)$ para $\sigma$ -- y aplica la capa fisica no entrenable de la Eq. 27: la sub-red $\text{cKAN}^\sigma$ no predice $\Delta p,\Delta q$ directamente, sino $\tilde G$ (a traves de un `softplus` que garantiza $\tilde G>0$), que junto con el escalar entrenable $\tilde R$ construye $\tilde K=\tilde R\tilde G$ y produce

$$\Delta p=\tilde K(\Delta\varepsilon_v-\Delta\varepsilon^{v,p}_{\text{pred}}),\qquad \Delta q=3\tilde G(\Delta\varepsilon_q-\Delta\gamma^p_{\text{pred}})$$

usando la deformacion plastica **predicha por cKAN$^{\varepsilon^p}$ en el mismo forward pass** (no la del dataset), que es precisamente lo que acopla fisicamente las tres salidas durante el entrenamiento.

In [ ]:
class ChebyKANLayer(nn.Module):
    """phi(x) = sum_i c_i T_i(x), polinomios de Chebyshev de primera especie (Eq. 25)."""

    def __init__(self, in_dim, out_dim, degree):
        super().__init__()
        self.degree = degree
        std = 1.0 / (in_dim * (degree + 1)) ** 0.5
        self.coeffs = nn.Parameter(std * torch.randn(in_dim, out_dim, degree + 1))

    def forward(self, x):
        x = torch.tanh(x)                       # dominio de Chebyshev: [-1, 1]
        Ts = [torch.ones_like(x), x]
        for _ in range(2, self.degree + 1):
            Ts.append(2 * x * Ts[-1] - Ts[-2])   # T_n = 2x T_{n-1} - T_{n-2}
        T = torch.stack(Ts, dim=-1)              # (batch, in_dim, degree+1)
        return torch.einsum('bid,iod->bo', T, self.coeffs)


class AugmentedChebyKANLayer(nn.Module):
    """phi(x) = Gamma(x) + Chebyshev(x), con Gamma(x) = omega * ReLU(x)  (Eq. 29)."""

    def __init__(self, in_dim, out_dim, degree):
        super().__init__()
        self.degree = degree
        std = 1.0 / (in_dim * (degree + 1)) ** 0.5
        self.coeffs = nn.Parameter(std * torch.randn(in_dim, out_dim, degree + 1))
        self.omega = nn.Parameter(0.1 * torch.randn(in_dim, out_dim))

    def forward(self, x):
        xn = torch.tanh(x)
        relu_term = torch.relu(xn) @ self.omega
        Ts = [torch.ones_like(xn), xn]
        for _ in range(2, self.degree + 1):
            Ts.append(2 * xn * Ts[-1] - Ts[-2])
        T = torch.stack(Ts, dim=-1)
        cheb_term = torch.einsum('bid,iod->bo', T, self.coeffs)
        return relu_term + cheb_term


class ChebyKAN(nn.Module):
    """Red KAN completa: composicion de capas cKAN (Eq. 20-21)."""

    def __init__(self, dims, degree, augmented_last=False):
        super().__init__()
        layers = []
        for i in range(len(dims) - 1):
            is_last = (i == len(dims) - 2)
            Layer = AugmentedChebyKANLayer if (augmented_last and is_last) else ChebyKANLayer
            layers.append(Layer(dims[i], dims[i + 1], degree))
        self.layers = nn.ModuleList(layers)

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x


class EPiCKAN(nn.Module):
    """cKAN^e + cKAN^(eps^p) + cKAN^sigma (predice G_tilde) + capa fisica no entrenable (Eq. 27)."""

    def __init__(self, n_in, degree_e=3, degree_ep=4, degree_sigma=4, hid=20,
                 G_scale=2.0e4, e_scale=1e-3, ep_scale=1e-3):
        super().__init__()
        self.kan_e = ChebyKAN([n_in, hid, 1], degree_e)                                   # (L,N,k)=(2,20,3)
        self.kan_ep = ChebyKAN([n_in, hid, hid, 2], degree_ep)                            # (3,20,4)
        self.kan_sigma = ChebyKAN([n_in, hid, hid, 1], degree_sigma, augmented_last=True)  # (3,20,4)
        self.R_tilde = nn.Parameter(torch.tensor(float(R_const)))                          # escalar Eq. 27
        self.G_scale, self.e_scale, self.ep_scale = G_scale, e_scale, ep_scale

    def forward(self, x_norm, dev, deq):
        de = self.kan_e(x_norm) * self.e_scale
        eps_p = self.kan_ep(x_norm) * self.ep_scale             # [Delta eps^v,p, Delta gamma^p]
        G_tilde = F.softplus(self.kan_sigma(x_norm)) * self.G_scale
        K_tilde = self.R_tilde * G_tilde
        deve = dev - eps_p[:, 0:1]
        deqe = deq - eps_p[:, 1:2]
        dp = K_tilde * deve
        dq = 3.0 * G_tilde * deqe
        return de, eps_p, torch.cat([dp, dq], dim=1)            # Eq. 27


model = EPiCKAN(n_in=9)
n_params = sum(p.numel() for p in model.parameters())
print('EPi-cKAN parametros entrenables:', n_params)

## 4. Funcion de costo fisica (Eq. 26, 28) y entrenamiento

$CF=CF^e+CF^{\varepsilon^p}+CF^\sigma$, con cada termino normalizado por la desviacion tipica del target correspondiente (necesario porque $\Delta e$ y $\Delta\varepsilon^p$ son $\sim10^{-4}$ mientras que $\Delta p,\Delta q$ son de decenas a cientos de kPa; el paper resuelve el mismo problema de escala normalizando todas las variables por min-max, Tabla 1c). El termino $CF^\sigma$ usa $\tilde C:(\Delta\varepsilon-\Delta\varepsilon^p_{\text{pred}})$, es decir, la deformacion plastica **predicha**, tal como exige la Eq. 28. Se entrena con Adam y `lr=1e-3`, igual que en el paper (Seccion 4.1), aunque con muchas menos epocas y datos que las 20 000 epocas / 187 000 muestras del paper, para mantener el tiempo de ejecucion razonable en un cuaderno.

In [ ]:
def physics_loss(model, Xraw, Xnorm, Y):
    dev, deq = Xraw[:, 7:8], Xraw[:, 8:9]
    de_pred, epsp_pred, dsigma_pred = model(Xnorm, dev, deq)
    Le = torch.mean(((de_pred - Y[:, 0:1]) / target_std_t[0]) ** 2)
    Lep = torch.mean(((epsp_pred - Y[:, 1:3]) / target_std_t[1:3]) ** 2)
    Ls = torch.mean(((dsigma_pred - Y[:, 3:5]) / target_std_t[3:5]) ** 2)
    return Le + Lep + Ls, Le.item(), Lep.item(), Ls.item()


optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
n_epochs = 6000
history = []
for epoch in range(n_epochs):
    model.train()
    optimizer.zero_grad()
    loss, le, lep, ls = physics_loss(model, Xt_train, Xnt_train, Yt_train)
    loss.backward()
    optimizer.step()
    history.append(loss.item())
    if epoch % 1000 == 0 or epoch == n_epochs - 1:
        model.eval()
        with torch.no_grad():
            vloss, *_ = physics_loss(model, Xt_val, Xnt_val, Yt_val)
        print(f'epoch {epoch:5d} | CF={loss.item():.4e} (e={le:.3e} ep={lep:.3e} sigma={ls:.3e}) '
              f'| CF_val={vloss.item():.4e}')

assert not np.isnan(loss.item()), 'NaN en la perdida!'
print('Entrenamiento terminado, sin NaN.')

plt.figure(figsize=(6, 4))
plt.semilogy(history)
plt.xlabel('epoca'); plt.ylabel('CF (log)'); plt.title('Evolucion de la funcion de costo (cf. Fig. 8 del paper)')
plt.tight_layout(); plt.show()

## 5. Resultados: error relativo L2 en el conjunto de evaluacion (estilo Tabla 3)

Igual que en la Eq. 31 del paper, se calcula el error relativo $\mathcal{L}^2$ de un solo paso (sin *rollout*) sobre el 15% del dataset reservado como conjunto de evaluacion, separado por sub-red.

In [ ]:
def rel_l2(pred, true):
    return 100 * np.linalg.norm(pred - true) / (np.linalg.norm(true) + 1e-12)


model.eval()
with torch.no_grad():
    de_p, epsp_p, dsig_p = model(Xnt_test, Xt_test[:, 7:8], Xt_test[:, 8:9])
de_p, epsp_p, dsig_p = de_p.numpy(), epsp_p.numpy(), dsig_p.numpy()

err_e = rel_l2(de_p, Y_test[:, 0:1])
err_ep = rel_l2(epsp_p, Y_test[:, 1:3])
err_sigma = rel_l2(dsig_p, Y_test[:, 3:5])

print('Errores relativos L2 (un paso, conjunto de evaluacion, estilo Tabla 3 del paper):')
print(f'  L2_e     (Delta e)                      = {err_e:6.2f}%')
print(f'  L2_eps^p (Delta eps^v,p, Delta gamma^p)  = {err_ep:6.2f}%')
print(f'  L2_sigma (Delta p, Delta q)              = {err_sigma:6.2f}%')
print('\n(Referencia, Tabla 3 del paper con red completa: L2_e=0.32%, L2_eps^p=2.55%, L2_sigma=0.13%)')

## 6. Prueba de trayectoria ciega (blind strain-controlled loading path), Sec. 4.2 del paper

Se reproduce el bucle de actualizacion de tensiones (Stage 1-5, Sec. 4.2 del paper) para una trayectoria axisimetrica **no vista en el entrenamiento**: $\xi=-2.5$, $p^{in}=375$ kPa, $e^{in}=0.64$, deformacion axial final $\varepsilon_{33}=0.07$ en 100 pasos iguales -- los mismos valores que usa el paper en sus Figuras 9 y 13(c,f).

**Stage 1**: estado inicial $e^0=e^{in}$, $p^0=p^{in}$, $q^0=0$, deformaciones nulas. **Stage 2**: incremento de deformacion fijo por paso ($\Delta\varepsilon_{33}=\varepsilon_{33}^{\text{final}}/n_{\text{step}}$). **Stage 3**: la red predice $(\Delta e,\Delta\varepsilon^{v,p},\Delta\gamma^p,\Delta p,\Delta q)$ a partir del estado actual. **Stage 4**: se actualiza el estado (Eq. 30). **Stage 5**: se repite hasta completar los 100 pasos. La red nunca ve el estado real (WG); cada paso se alimenta de sus **propias predicciones anteriores** -- una prueba mucho mas exigente que el error de un solo paso de la Seccion 5.

In [ ]:
xi_test, p_in_test, e_in_test, n_steps_test = -2.5, 375.0, 0.64, 100
recs_true = simulate_path(xi_test, p_in_test, e_in_test, n_steps=n_steps_test, random_incr=False)

d33 = 0.07 / n_steps_test
d11 = d33 / xi_test
dev_step = 2 * d11 + d33
deq_step = (2.0 / 3.0) * (d33 - d11)

state_pred = dict(e=e_in_test, p=p_in_test, q=0.0, ev=0.0, eq=0.0, evp=0.0, gammap=0.0)
pred_records = [state_pred.copy()]
with torch.no_grad():
    for _ in range(n_steps_test):
        x = np.array([[state_pred['e'], state_pred['p'], state_pred['q'], state_pred['ev'],
                        state_pred['eq'], state_pred['evp'], state_pred['gammap'],
                        dev_step, deq_step]], dtype=np.float32)
        xt, xnt = torch.tensor(x), torch.tensor(normalize_x(x))
        de_pred, epsp_pred, dsigma_pred = model(xnt, xt[:, 7:8], xt[:, 8:9])
        # Stage 4: actualizacion del estado (Eq. 30), realimentando la propia prediccion
        state_pred['e'] += de_pred.item()
        state_pred['evp'] += epsp_pred[0, 0].item()
        state_pred['gammap'] += epsp_pred[0, 1].item()
        state_pred['p'] += dsigma_pred[0, 0].item()
        state_pred['q'] += dsigma_pred[0, 1].item()
        state_pred['ev'] += dev_step
        state_pred['eq'] += deq_step
        pred_records.append(state_pred.copy())

eps33 = np.linspace(0, 0.07, n_steps_test + 1)
q_true = np.array([r['q'] for r in recs_true]); q_pred = np.array([r['q'] for r in pred_records])
p_true = np.array([r['p'] for r in recs_true]); p_pred = np.array([r['p'] for r in pred_records])
e_true = np.array([r['e'] for r in recs_true]); e_pred = np.array([r['e'] for r in pred_records])
evp_true = np.array([r['evp'] for r in recs_true]); evp_pred = np.array([r['evp'] for r in pred_records])
gammap_true = np.array([r['gammap'] for r in recs_true]); gammap_pred = np.array([r['gammap'] for r in pred_records])

print(f'Trayectoria ciega xi={xi_test}, p_in={p_in_test} kPa, e_in={e_in_test}:')
print(f'  L2 error q = {rel_l2(q_pred, q_true):.2f}%   L2 error p = {rel_l2(p_pred, p_true):.2f}%')
print(f'  L2 error e = {rel_l2(e_pred, e_true):.2f}%   L2 error eps^v,p = {rel_l2(evp_pred, evp_true):.2f}%'
      f'   L2 error gamma^p = {rel_l2(gammap_pred, gammap_true):.2f}%')
print('(Referencia paper, Tabla 4, xi=-2.5: EPi-cKAN=1.26%, EPNN=7.11%, en Delta q)')
assert not np.any(np.isnan(q_pred)), 'NaN en la trayectoria ciega!'

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes[0, 0].plot(eps33, evp_true, label='WG (ref.)'); axes[0, 0].plot(eps33, evp_pred, '--', label='EPi-cKAN')
axes[0, 0].set_xlabel(r'$\varepsilon_{33}$'); axes[0, 0].set_ylabel(r'$\varepsilon^{v,p}$'); axes[0, 0].legend()
axes[0, 1].plot(eps33, gammap_true, label='WG'); axes[0, 1].plot(eps33, gammap_pred, '--', label='EPi-cKAN')
axes[0, 1].set_xlabel(r'$\varepsilon_{33}$'); axes[0, 1].set_ylabel(r'$\gamma^p$'); axes[0, 1].legend()
axes[0, 2].plot(eps33, e_true, label='WG'); axes[0, 2].plot(eps33, e_pred, '--', label='EPi-cKAN')
axes[0, 2].set_xlabel(r'$\varepsilon_{33}$'); axes[0, 2].set_ylabel('e'); axes[0, 2].legend()
axes[1, 0].plot(eps33, p_true, label='WG'); axes[1, 0].plot(eps33, p_pred, '--', label='EPi-cKAN')
axes[1, 0].set_xlabel(r'$\varepsilon_{33}$'); axes[1, 0].set_ylabel('p (kPa)'); axes[1, 0].legend()
axes[1, 1].plot(eps33, q_true, label='WG'); axes[1, 1].plot(eps33, q_pred, '--', label='EPi-cKAN')
axes[1, 1].set_xlabel(r'$\varepsilon_{33}$'); axes[1, 1].set_ylabel('q (kPa)'); axes[1, 1].legend()
axes[1, 2].plot(p_true, q_true, label='WG'); axes[1, 2].plot(p_pred, q_pred, '--', label='EPi-cKAN')
axes[1, 2].set_xlabel('p (kPa)'); axes[1, 2].set_ylabel('q (kPa)'); axes[1, 2].legend()
plt.suptitle(f'Trayectoria ciega xi={xi_test} (cf. Fig. 9 del paper)')
plt.tight_layout(); plt.show()

## 7. Robustez frente a ruido (5%), cf. Fig. 14 del paper

El paper anade ruido blanco (media 0, desviacion 5% del valor) a las tensiones de entrenamiento y muestra que EPi-cKAN degrada mucho menos que el EPNN (version MLP). Aqui entrenamos una segunda instancia del mismo modelo con 5% de ruido en $\Delta p,\Delta q$ y comparamos su error en la trayectoria ciega frente al modelo sin ruido.

In [ ]:
rng_noise = np.random.RandomState(2)
Y_train_noisy = Y_train.copy()
noise = rng_noise.normal(0, 0.05, size=Y_train[:, 3:5].shape) * Y_train[:, 3:5]
Y_train_noisy[:, 3:5] = Y_train[:, 3:5] + noise
Yt_train_noisy = torch.tensor(Y_train_noisy)

model_noisy = EPiCKAN(n_in=9)
opt_noisy = torch.optim.Adam(model_noisy.parameters(), lr=1e-3)
n_epochs_noisy = 2000
for epoch in range(n_epochs_noisy):
    opt_noisy.zero_grad()
    loss_n, *_ = physics_loss(model_noisy, Xt_train, Xnt_train, Yt_train_noisy)
    loss_n.backward()
    opt_noisy.step()
print(f'Modelo con ruido (5% en Delta sigma), CF final = {loss_n.item():.4e}')
assert not np.isnan(loss_n.item()), 'NaN en el modelo con ruido!'

state_pred_n = dict(e=e_in_test, p=p_in_test, q=0.0, ev=0.0, eq=0.0, evp=0.0, gammap=0.0)
q_pred_n = [0.0]
model_noisy.eval()
with torch.no_grad():
    for _ in range(n_steps_test):
        x = np.array([[state_pred_n['e'], state_pred_n['p'], state_pred_n['q'], state_pred_n['ev'],
                        state_pred_n['eq'], state_pred_n['evp'], state_pred_n['gammap'],
                        dev_step, deq_step]], dtype=np.float32)
        xt, xnt = torch.tensor(x), torch.tensor(normalize_x(x))
        de_pred, epsp_pred, dsigma_pred = model_noisy(xnt, xt[:, 7:8], xt[:, 8:9])
        state_pred_n['e'] += de_pred.item()
        state_pred_n['evp'] += epsp_pred[0, 0].item()
        state_pred_n['gammap'] += epsp_pred[0, 1].item()
        state_pred_n['p'] += dsigma_pred[0, 0].item()
        state_pred_n['q'] += dsigma_pred[0, 1].item()
        state_pred_n['ev'] += dev_step
        state_pred_n['eq'] += deq_step
        q_pred_n.append(state_pred_n['q'])
q_pred_n = np.array(q_pred_n)

print(f'Error L2 en q con entrenamiento ruidoso (5%): {rel_l2(q_pred_n, q_true):.2f}%  '
      f'(vs {rel_l2(q_pred, q_true):.2f}% sin ruido)')

plt.figure(figsize=(6, 4))
plt.plot(eps33, q_true, label='WG (ref.)')
plt.plot(eps33, q_pred, '--', label='EPi-cKAN (sin ruido)')
plt.plot(eps33, q_pred_n, ':', label='EPi-cKAN (5% ruido en entrenamiento)')
plt.xlabel(r'$\varepsilon_{33}$'); plt.ylabel('q (kPa)'); plt.legend()
plt.title('Robustez frente a ruido (cf. Fig. 14 del paper)')
plt.tight_layout(); plt.show()

## 8. Verificacion termodinamica a posteriori (cf. Fig. 15 del paper)

Al igual que el paper (Sec. 5, "we note that although the cost function does not explicitly enforce thermodynamic constraints, compliance with the first and second laws is verified a posteriori"), calculamos la energia elastica acumulada $\Delta W_{el}=\sum(p\,\Delta\varepsilon^e_v+q\,\Delta\varepsilon^e_q)$ y la disipacion plastica acumulada $\Delta D=\sum(p\,\Delta\varepsilon^{v,p}+q\,\Delta\gamma^p)$ a lo largo de la trayectoria ciega, comparando el modelo WG (referencia) con las predicciones de EPi-cKAN. La segunda ley exige $\Delta D\ge0$ en todo momento.

In [ ]:
def thermo_quantities(records):
    dWel, dD = [0.0], [0.0]
    for i in range(1, len(records)):
        s0, s1 = records[i - 1], records[i]
        dev_e = (s1['ev'] - s0['ev']) - (s1['evp'] - s0['evp'])
        deq_e = (s1['eq'] - s0['eq']) - (s1['gammap'] - s0['gammap'])
        p_avg, q_avg = 0.5 * (s0['p'] + s1['p']), 0.5 * (s0['q'] + s1['q'])
        dWel.append(dWel[-1] + p_avg * dev_e + q_avg * deq_e)
        devp, deqp = s1['evp'] - s0['evp'], s1['gammap'] - s0['gammap']
        dD.append(dD[-1] + p_avg * devp + q_avg * deqp)
    return np.array(dWel), np.array(dD)


dWel_true, dD_true = thermo_quantities(recs_true)
dWel_pred, dD_pred = thermo_quantities(pred_records)

print(f'Energia elastica acumulada:     WG={dWel_true[-1]:.2f}  EPi-cKAN={dWel_pred[-1]:.2f}  '
      f'(L2 error {rel_l2(dWel_pred, dWel_true):.2f}%)')
print(f'Disipacion plastica acumulada:  WG={dD_true[-1]:.2f}  EPi-cKAN={dD_pred[-1]:.2f}  '
      f'(L2 error {rel_l2(dD_pred, dD_true):.2f}%)')
print(f'Disipacion siempre no-negativa (2da ley): {bool(np.all(dD_pred >= -1e-6))}')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(eps33, dWel_true, label='WG'); axes[0].plot(eps33, dWel_pred, '--', label='EPi-cKAN')
axes[0].set_xlabel(r'$\varepsilon_{33}$'); axes[0].set_ylabel(r'$\Delta W_{el}$ acumulada'); axes[0].legend()
axes[1].plot(eps33, dD_true, label='WG'); axes[1].plot(eps33, dD_pred, '--', label='EPi-cKAN')
axes[1].set_xlabel(r'$\varepsilon_{33}$'); axes[1].set_ylabel(r'$\Delta D$ acumulada'); axes[1].legend()
plt.suptitle('Verificacion termodinamica a posteriori (cf. Fig. 15 del paper)')
plt.tight_layout(); plt.show()

print(f'\nTiempo total de ejecucion del cuaderno: {time.time() - t_start:.0f} s')

## Nota honesta sobre los resultados

**Lo que se obtuvo al ejecutar este cuaderno** (semillas fijas, resultados reproducibles, ~7-8 minutos en CPU):

| Metrica | Este cuaderno | Paper (red completa) |
|---|---|---|
| $\mathcal{L}^2_e$ (un paso) | 0.46% | 0.32% |
| $\mathcal{L}^2_{\varepsilon^p}$ (un paso) | 6.34% | 2.55% |
| $\mathcal{L}^2_\sigma$ (un paso) | 1.47% | 0.13% |
| $\mathcal{L}^2$ de $\Delta q$, trayectoria ciega $\xi=-2.5$ | 29.29% | 1.26% |
| $\mathcal{L}^2$ de $\Delta q$ con 5% de ruido, $\xi=-2.5$ | 16.71%* | 1.52% |
| Disipacion plastica siempre no-negativa (2da ley) | Si | Si |
| Error $\mathcal{L}^2$ en disipacion plastica acumulada | 11.95% | -- |
| Error $\mathcal{L}^2$ en energia elastica acumulada | 50.47% | -- |

\*En nuestra ejecucion, el modelo entrenado con ruido dio *menor* error de rollout que el limpio (16.71% vs 29.29%) -- lo contrario del efecto esperado. Con una red de solo ~7000 parametros, ~9000 muestras y 2000-6000 epocas, la trayectoria ciega (100 pasos autoregresivos, alimentando cada paso con las propias predicciones del anterior) resulta muy sensible a la inicializacion aleatoria de cada entrenamiento; no debe leerse como una replica fiable del efecto de robustez del paper, solo como evidencia cualitativa de que **el entrenamiento con datos ruidosos no diverge ni degrada catastroficamente**.

**Lectura de la tabla**: el error de un solo paso (filas 1-3) es del mismo orden de magnitud que el del paper -- razonable para una red \~2 ordenes de magnitud mas pequena en parametros efectivos de entrenamiento, con \~65 veces menos datos y \~3-10 veces menos epocas. El error de la **trayectoria ciega** (fila 4), en cambio, es sustancialmente mayor que el 1.26% del paper: al ser un *rollout* autoregresivo de 100 pasos, cualquier sesgo sistematico del modelo simplificado se acumula paso a paso -- la precision de un paso no garantiza precision tras 100 iteraciones, un fenomeno real y bien conocido en la evaluacion de modelos sustitutos autoregresivos, no un artefacto de la implementacion. La verificacion termodinamica confirma que la **disipacion plastica se mantiene siempre no-negativa** (igual que en la Fig. 15 del paper, con solo 11.95% de error acumulado), mientras que la energia elastica acumulada se desvia mas (~50%) -- consistente con que ese calculo es una diferencia de dos cantidades similares ($\Delta\varepsilon_v-\Delta\varepsilon^{v,p}$) y por tanto amplifica el mismo error de rollout ya visible en $p,q$.

**Simplificaciones respecto al paper** (necesarias para que el modelo fisico de referencia y el entrenamiento cupieran en un cuaderno ejecutable en minutos sobre CPU):

1. **Reduccion tensorial**: se trabaja en el subespacio de invariantes axisimetricos $(e,p,q,\varepsilon_v,\varepsilon_q,\varepsilon^{v,p},\gamma^p)$ en vez del tensor de deformacion/tension completo de 6 componentes. Es exactamente el subespacio en el que el paper reporta sus resultados (Figs. 9-14), pero para el *entrenamiento* el paper usa trayectorias 3D generales, no solo axisimetricas.
2. **Dilatancia simplificada**: las Eqs. 12-13 del paper (mezcla de friccion movilizada en compresion/extension triaxial via el angulo de Lode $t$ y la constante $\mu$) no se pudieron reconstruir sin ambiguedad a partir del texto extraido del PDF (fracciones y subindices con perdida de formato en la extraccion). Se sustituyeron por una relacion de dilatancia tipo Rowe/estado critico estandar, $N=n_d(\eta-M_{cs})$ con $\eta=q/p$, atenuada por un factor empirico $n_d=0.2$ para evitar una sobre-reaccion numerica del esquema explicito (con $n_d=1$ aparecen oscilaciones no fisicas de $p$ en la trayectoria simulada). La ley de endurecimiento del angulo de friccion movilizado (Eq. 14b) si se implemento exactamente como en el paper.
3. **Integracion explicita vs. implicita**: el paper integra el modelo WG con Euler regresivo y *return mapping* (algoritmo estandar en plasticidad computacional); aqui se usa un esquema explicito con multiplicador plastico consistente y sub-pasos finos, mas simple de implementar de forma robusta pero con distinta acumulacion de error numerico.
4. **Escala del modelo y del entrenamiento**: ~6900-7000 parametros, ~9000 muestras de entrenamiento y 2000-6000 epocas en CPU, frente a los 13 920-14 560 parametros, 187 000 muestras y 20 000 epocas en GPU (RTX A6000) del paper (Tabla 2, Seccion 4.1).
5. **Termino $\Delta\gamma^p$**: se identifica $\gamma^p\equiv|\varepsilon_q^p|$ (deformacion plastica de corte escalar), valido en el subespacio axisimetrico con un unico componente desviador activo, en vez del invariante tensorial general de la Eq. 15.

Pese a estas simplificaciones, el cuaderno reproduce fielmente los elementos **arquitectonicos y de perdida** que constituyen la contribucion central del paper: la capa cKAN con polinomios de Chebyshev (Eq. 25), la base aumentada Gamma+Chebyshev (Eq. 29), y sobre todo la **capa fisica no entrenable** que impone la relacion elastica hipoelastica exacta del modelo WG (Eq. 27) usando la deformacion plastica *predicha* en el mismo forward pass -- el mecanismo que hace a EPi-cKAN "informado por la fisica" y no un simple regresor de tres salidas independientes.